In [ ]:
from pathlib import Path
from matplotlib import pyplot as plt 
import arviz as az

from autumn.projects.sm_covid2.common_school.calibration import get_bcm_object

In [ ]:
param_names = {
    "contact_rate": "transmission risk", 
    "age_stratification.ifr.multiplier": "IFR multiplier", 
    "school_multiplier": "school uncertainty multiplier", 
    "mobility.unesco_partial_opening_value": "partial opening attendence"
}
param_list = [v for k, v in param_names.items()]
analysis_folder = Path.cwd() / "33489767_test_full_analysis_24Jan2024_main"
iso3_list = ["MAR", "IDN", "GBR"]

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde
import estival.priors as esp
from numpyro import distributions as dist

def convert_prior_to_numpyro(prior):
    """
    Converts a given custom prior to a corresponding Numpyro distribution and its bounds based on its type.

    Args:
        prior: A custom prior object.

    Returns:
        A tuple of (Numpyro distribution, bounds).
    """
    if isinstance(prior, esp.UniformPrior):
        return dist.Uniform(low=prior.start, high=prior.end), (prior.start, prior.end)
    elif isinstance(prior, esp.TruncNormalPrior):
        return dist.TruncatedNormal(
            loc=prior.mean,
            scale=prior.stdev,
            low=prior.trunc_range[0],
            high=prior.trunc_range[1],
        ), (prior.trunc_range[0], prior.trunc_range[1])
    elif isinstance(prior, esp.GammaPrior):
        rate = 1.0 / prior.scale
        return dist.Gamma(concentration=prior.shape, rate=rate), None
    elif isinstance(prior, esp.BetaPrior):
        return dist.Beta(concentration1=prior.a, concentration0=prior.b), (0, 1)
    else:
        raise TypeError(f"Unsupported prior type: {type(prior).__name__}")



def plot_post_prior_comparison(idata, priors, params_name):
    """
    Plot comparison of model posterior outputs against priors.

    Args:
        idata: Arviz inference data from calibration.
        priors: Dictionary of custom prior objects.
        params_name: Dictionary mapping parameter names to descriptive titles.

    Returns:
        The figure object.
    """
    # Filter priors to exclude those containing '_dispersion'
    req_vars = [
        var
        for var in priors.keys()
        if "_dispersion" not in var and var != "contact_reduction"
    ]
    num_vars = len(req_vars)
    num_rows = (num_vars + 1) // 2  # Ensure even distribution across two columns

    # Set figure size to match A4 page width (8.27 inches) in portrait mode and adjust height based on rows
    fig, axs = plt.subplots(
        num_rows, 2, figsize=(28, 6.2 * num_rows)
    )  # A4 width in portrait mode
    axs = axs.ravel()

    for i_ax, ax in enumerate(axs):
        if i_ax < num_vars:
            var_name = req_vars[i_ax]
            posterior_samples = idata.posterior[var_name].values.flatten()
            low_post = np.min(posterior_samples)
            high_post = np.max(posterior_samples)
            x_vals_posterior = np.linspace(low_post, high_post, 100)

            # Use gaussian_kde to estimate the posterior density
            post_kde = gaussian_kde(posterior_samples)
            posterior_density = post_kde(x_vals_posterior)

            # Convert the prior to a Numpyro distribution
            numpyro_prior, prior_bounds = convert_prior_to_numpyro(priors[var_name])
            if prior_bounds:
                low_prior, high_prior = prior_bounds
                x_vals_prior = np.linspace(low_prior, high_prior, 100)
            else:
                x_vals_prior = (
                    x_vals_posterior  # Fallback if no specific prior bounds are given
                )

            # Compute the prior density using Numpyro
            prior_density = np.exp(numpyro_prior.log_prob(x_vals_prior))

            # Plot the prior density
            ax.fill_between(
                x_vals_prior,
                prior_density,
                color="k",
                alpha=0.2,
                linewidth=2,
                label="Prior",
            )
            ax.fill_between(
                x_vals_posterior, 0, posterior_density, color="b", alpha=0.3, label="Posterior",
            )  # Fill under posterior

            # Set the title using the descriptive name from params_name
            title = params_name.get(
                var_name, var_name
            )  # Use var_name if not in params_name
            ax.set_title(title, fontsize=34, fontname="Arial")  # Set title to Arial 30
            ax.tick_params(axis="both", labelsize=24)

            # Add legend to the first subplot
            if i_ax == 0:
                ax.legend(fontsize=24)
        else:
            ax.axis("off")  # Turn off empty subplots if the number of req_vars is odd

    # Adjust padding and spacing
    plt.tight_layout(
        h_pad=1.0, w_pad=5
    )  # Increase padding between plots for better fit
    return fig



In [ ]:
folderpath = Path.cwd() / "posterior_plots"
for iso3 in iso3_list:   

    idata = az.from_netcdf(analysis_folder / iso3 / "idata.nc")
    burnt_idata = idata.sel(draw=range(25000, 30000))    

    bcm = get_bcm_object(iso3, 'main')
    priors = bcm.priors
    selected_priors = {k: v for k,v in priors.items() if k in param_names}
    fig = plot_post_prior_comparison(burnt_idata, selected_priors, param_names)
    plt.savefig(folderpath / f"{iso3}_posteriors.png", dpi=300)  

    renamed_burnt_idata = burnt_idata.rename(param_names)
    az.plot_pair(renamed_burnt_idata, var_names=param_list)
    plt.savefig(folderpath / f"{iso3}_pairs.png", dpi=300)  

